In [1]:
import streamlit as st
import duckdb
import pandas as pd
import requests

In [2]:
# get parquet files from Deadlock API
manifest = requests.get("https://data.deadlock-api.com/v1/manifest.json").json()
base = manifest["public_url"]

def urls(table: str, limit: int = 50) -> list[str]:
    t = manifest["tables"][table]
    gen = t.get("generation", 0)
    file_list = [f"{base}/{f['key']}" for f in t["files"]
        if t["policy"] == "snapshot" or f.get("generation", 0) == gen]
    return file_list[-limit:]

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

con.execute("SET threads = 2;")
con.execute("SET memory_limit = '4GB';")
con.execute("SET preserve_insertion_order = false;")

file_urls = urls("match_player", limit=30)
con.execute(f"CREATE VIEW match_player AS SELECT * FROM read_parquet({file_urls!r}, union_by_name = true)")

In [ ]:
pd.set_option("display.max_colwidth", None)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 1000)
pd.set_option("display.expand_frame_repr", False)

In [4]:
extracted_data = con.execute("""
SELECT match_id, player_slot, team, won, COLUMNS('^stats\\.'), game_mode
FROM match_player
WHERE game_mode NOT IN ('StreetBrawl', 'Training', 'Tutorial')
LIMIT 360
""").df()


player_timelines = []

for _, row in extracted_data.iterrows():
    
    # gathers timestamps from the stats column
    timestamps = row["stats.time_stamp_s"]
    
    if timestamps is None:
        continue
    
    # gather static information from the matches
    timeline = pd.DataFrame({"timestamp": timestamps})
    timeline["match_id"] = row["match_id"]
    timeline["team"] = row["team"]
    timeline["won"] = row["won"]
    
    for col in extracted_data.columns:
        
        # gathers the stats at the given timestamp for the match
        if col.startswith("stats.") and col != "stats.time_stamp_s":
            values = row[col]
            
            if values is not None and len(values) == len(timestamps):
                timeline[col.replace("stats.", "")] = values
                
    player_timelines.append(timeline)


all_timelines = pd.concat(player_timelines, ignore_index=True)
all_timelines["minutes"] = all_timelines["timestamp"] / 60
display(all_timelines)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,timestamp,match_id,team,won,net_worth,gold_player,gold_player_orbs,gold_lane_creep_orbs,gold_neutral_creep_orbs,gold_boss,gold_boss_orb,gold_treasure,gold_denied,gold_death_loss,gold_lane_creep,gold_neutral_creep,kills,deaths,assists,creep_kills,neutral_kills,possible_creeps,creep_damage,player_damage,neutral_damage,boss_damage,denies,player_healing,ability_points,self_healing,player_barriering,teammate_healing,teammate_barriering,self_damage,bullet_kills,melee_kills,ability_kills,headshot_kills,player_damage_taken,max_health,weapon_power,tech_power,shots_hit,shots_missed,damage_absorbed,absorption_provided,hero_bullets_hit,hero_bullets_hit_crit,heal_prevented,heal_lost,damage_mitigated,level,gold_source_assists_gold,gold_source_team_bonus_gold,gold_source_breakable_gold,gold_source_ability_assassinate_gold,gold_source_item_trophy_collector_gold,gold_source_item_cultist_sacrifice_gold,gold_source_item_goose_egg_gold,minutes
0,180,92214755,Team0,False,1526,0,0,365,0,0,0,0,0,0,561,0,0,0,0,8,0,10,1818,836,0,0,0,0,2,0,0,0,0,0,0,0,0,0,705,829,17,1,200,340,302,302,161,59,0,0,657,4,0,0,0,0,0,0,0,3.000000
1,360,92214755,Team0,False,2918,0,0,900,0,0,0,0,0,0,1418,0,0,0,0,16,0,32,4436,1372,0,0,0,0,3,0,0,0,0,0,0,0,0,0,1061,895,17,2,471,690,497,497,314,110,0,0,1035,6,0,0,0,0,0,0,0,6.000000
2,540,92214755,Team0,False,5260,817,0,1445,0,127,0,0,29,0,2063,96,1,1,0,25,2,46,6488,3584,292,700,1,0,6,0,0,0,0,0,0,0,1,0,2417,1027,20,28,751,1022,616,616,498,162,0,0,1303,10,0,83,0,0,0,0,0,9.000000
3,720,92214755,Team0,False,7363,819,0,2180,0,418,0,0,49,0,2964,140,1,2,0,35,3,62,9747,6139,568,1253,1,0,8,0,0,0,0,0,0,0,1,0,4173,1093,20,31,1000,1280,1190,1190,684,188,0,0,2141,12,0,151,42,0,0,0,0,12.000000
4,900,92214755,Team0,False,8400,950,0,2578,0,418,0,0,49,0,3317,140,1,3,1,39,3,67,11935,7523,568,1253,1,0,10,0,0,0,0,0,0,0,1,0,6442,1159,23,51,1142,1527,1672,1672,786,198,0,7,3494,14,76,306,42,0,0,0,0,15.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3682,1800,92214871,Team0,True,31552,6357,0,4074,0,3966,0,0,140,325,10456,1622,5,6,12,95,14,117,36305,18425,6323,1339,3,5674,26,5674,0,0,0,0,1,2,2,0,22515,3159,58,108,3600,4167,0,0,200,43,0,0,6111,30,2672,578,1268,0,2816,0,0,30.000000
3683,2100,92214871,Team0,True,43140,7600,0,6370,0,4166,0,987,140,325,14578,2672,6,6,14,133,15,160,49249,24094,11472,1339,3,6332,30,6332,548,0,0,0,1,2,3,0,24064,3529,76,121,4281,4890,548,548,221,46,0,0,7966,34,3172,578,1535,0,4239,0,0,35.000000
3684,2400,92214871,Team0,True,53059,9786,0,7442,0,5916,0,987,140,325,16823,3497,8,7,19,160,15,189,63310,32671,16900,8400,3,7711,32,7711,704,0,0,0,1,2,5,0,29554,3643,76,139,5097,5487,704,704,257,51,0,0,12930,36,4237,578,1535,400,5680,0,0,40.000000
3685,2700,92214871,Team0,True,61333,10015,0,7739,0,7376,0,5320,140,325,17338,3497,8,8,20,163,15,193,65552,34786,16900,25249,3,8513,32,8513,1206,0,0,0,1,2,5,0,34340,3653,54,168,6317,5554,1206,1206,269,52,0,0,17634,36,4432,578,1535,400,7120,0,0,45.000000


In [8]:
# exclude all columns not representing stats
stat_columns = [
    col for col in all_timelines.columns
    if col not in [
        "timestamp",
        "match_id",
        "team",
        "won",
        "minutes"
    ]
]

# create a simple variable for each stats column to more easily use SQL 
avg_columns = ", ".join(
    f'AVG("{col}") AS "{col}"' 
    for col in stat_columns
    )


team_timelines = con.execute(f"""
SELECT match_id,
team,
won,
timestamp,
minutes,
{avg_columns}
FROM all_timelines
GROUP BY match_id, team, won, timestamp, minutes
ORDER BY match_id, team, timestamp, minutes
""").df()

team_timelines.head(25)

,match_id,team,won,timestamp,minutes,net_worth,gold_player,gold_player_orbs,gold_lane_creep_orbs,gold_neutral_creep_orbs,gold_boss,gold_boss_orb,gold_treasure,gold_denied,gold_death_loss,gold_lane_creep,gold_neutral_creep,kills,deaths,assists,creep_kills,neutral_kills,possible_creeps,creep_damage,player_damage,neutral_damage,boss_damage,denies,player_healing,ability_points,self_healing,player_barriering,teammate_healing,teammate_barriering,self_damage,bullet_kills,melee_kills,ability_kills,headshot_kills,player_damage_taken,max_health,weapon_power,tech_power,shots_hit,shots_missed,damage_absorbed,absorption_provided,hero_bullets_hit,hero_bullets_hit_crit,heal_prevented,heal_lost,damage_mitigated,level,gold_source_assists_gold,gold_source_team_bonus_gold,gold_source_breakable_gold,gold_source_ability_assassinate_gold,gold_source_item_trophy_collector_gold,gold_source_item_cultist_sacrifice_gold,gold_source_item_goose_egg_gold
0,92214755,Team0,False,180,3.00,1697.666667,0.000000,0.0,347.000000,0.0,0.000000,0.0,0.000000,51.833333,0.000000,653.000000,41.000000,0.000000,0.000000,0.000000,8.000000,1.000000,9.500000,2466.000000,560.166667,148.666667,0.000000,1.000000,118.166667,2.000000,118.166667,50.333333,0.000000,50.333333,0.0,0.000000,0.000000,0.000000,0.000000,747.166667,911.333333,2.833333,7.833333,177.333333,211.500000,50.333333,50.333333,53.333333,13.666667,0.000000,0.166667,384.666667,4.000000,0.000000,0.000000,4.833333,0.0,0.000000,0.000000,0.000000
1,92214755,Team0,False,360,6.00,3256.666667,194.333333,0.0,798.833333,0.0,0.000000,0.0,0.000000,78.500000,0.000000,1470.500000,104.666667,0.333333,0.333333,0.333333,14.666667,2.500000,21.666667,5087.000000,1859.333333,393.500000,150.166667,1.500000,546.166667,3.333333,546.166667,132.666667,0.000000,82.833333,0.0,0.000000,0.000000,0.333333,0.000000,1815.666667,1059.333333,4.333333,14.166667,359.166667,452.166667,132.666667,132.666667,114.666667,28.333333,0.000000,1.833333,966.666667,6.500000,60.666667,0.000000,9.833333,0.0,0.000000,0.000000,0.000000
2,92214755,Team0,False,540,9.00,6073.666667,1083.000000,0.0,1326.000000,0.0,213.166667,0.0,0.000000,88.000000,7.666667,2206.166667,420.666667,1.166667,1.166667,1.166667,21.666667,6.333333,32.500000,7776.000000,3544.000000,1350.833333,696.666667,1.666667,1238.166667,6.500000,1159.166667,277.666667,79.000000,102.666667,0.0,0.000000,0.000000,1.166667,0.000000,3492.666667,1332.166667,4.833333,28.500000,530.166667,634.166667,277.666667,277.666667,171.333333,39.666667,0.000000,2.000000,1587.500000,10.500000,263.833333,55.000000,80.000000,0.0,9.333333,0.000000,0.000000
3,92214755,Team0,False,720,12.00,8595.000000,1361.500000,0.0,1731.833333,0.0,421.166667,0.0,0.000000,97.833333,52.666667,3028.333333,915.333333,1.500000,1.666667,1.500000,28.833333,12.500000,42.500000,10651.333333,5701.000000,3287.333333,1140.000000,1.833333,1872.500000,10.000000,1606.166667,503.500000,266.333333,258.500000,0.0,0.000000,0.333333,1.166667,0.000000,5090.333333,1671.000000,4.833333,47.000000,683.500000,759.166667,503.500000,503.500000,219.833333,45.833333,27.166667,3.833333,2244.833333,14.000000,350.000000,153.833333,291.666667,0.0,46.166667,0.000000,0.000000
4,92214755,Team0,False,900,15.00,10523.166667,1617.666667,0.0,2302.333333,0.0,421.166667,0.0,0.000000,97.833333,89.166667,3891.833333,1057.666667,1.833333,2.666667,2.666667,36.833333,13.666667,53.333333,13799.000000,7991.000000,3732.333333,1140.000000,1.833333,2928.833333,12.333333,2317.833333,675.500000,611.000000,341.833333,0.0,0.000000,0.333333,1.500000,0.000000,8104.500000,1959.666667,14.000000,58.166667,765.666667,927.333333,675.500000,675.500000,254.333333,50.666667,50.833333,9.666667,3215.666667,16.333333,459.166667,213.666667,308.333333,0.0,101.833333,0.000000,0.000000
5,92214755,Team0,False,1200,20.00,15389.666667,2012.000000,0.0,3438.500000,0.0,662.500000,0.0,0.000000,282.000000,299.000000,5429.000000,2142.833333,2.166667,3.500000,3.666667,48.833333,22.333333,70.000000,18614.833333,11120.833333,629